# 03 — Limpieza e ingeniería de variables para modelado

## ¿Qué voy a hacer en este notebook?

En la Parte 1 encontré que el fraude se concentra en `TRANSFER` y `CASH_OUT`, en montos altos y en operaciones que suelen vaciar la cuenta origen. Ahora voy a transformar esas observaciones en variables que un modelo pueda usar.

Mi objetivo todavía **no es entrenar**. Primero voy a:

1. Definir qué columnas puedo usar y cuáles debo excluir.
2. Crear variables derivadas con significado de negocio.
3. Separar entrenamiento, validación y prueba respetando el tiempo.
4. Preparar una muestra de entrenamiento que permita trabajar con el fuerte desbalance.

> Este modelo se plantea para priorizar revisión **después** de registrada la transacción, porque utilizo saldos posteriores. No lo presento como un bloqueo previo de pagos.

In [4]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'paysim.csv'
IMAGES_DIR = PROJECT_ROOT / 'images'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
for folder in [IMAGES_DIR, REPORTS_DIR, MODELS_DIR, OUTPUTS_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


## Carga controlada del dataset

Cargo solamente las columnas necesarias para esta fase. No necesito los identificadores de cuenta para modelar y evitarlos reduce memoria y el riesgo de que el modelo memorice cuentas concretas en vez de aprender patrones.

In [5]:
# Carga las funciones para leer el dataset y crear variables derivadas
from modeling_utils import engineer_features, load_transactions

# Lee el archivo CSV con las transacciones
raw = load_transactions(DATA_RAW)

# Muestra cuántas transacciones se cargaron
print(f'Transacciones cargadas: {len(raw):,}')

# Muestra cuántos fraudes reales contiene el dataset
print(f'Fraudes reales: {int(raw["isFraud"].sum()):,}')

# Visualiza las primeras filas para comprobar la estructura de los datos
raw.head()

Transacciones cargadas: 6,362,620
Fraudes reales: 8,213


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0


## Qué variables usaré y por qué

| Decisión | Columnas | Razón |
|---|---|---|
| Incluyo información original | `type`, `amount`, saldos antes/después | El EDA mostró diferencias por tipo, monto y saldo. |
| Creo señales explicables | `log_amount`, diferencias, errores, ratio y saldo cero | Traducen comportamiento económico a números comparables. |
| Excluyo identificadores | `nameOrig`, `nameDest` | Una cuenta concreta no es una regla generalizable de fraude. |
| Excluyo la alerta previa | `isFlaggedFraud` | La usaré como comparación, no como atajo para el nuevo modelo. |
| Excluyo tiempo como predictor | `step` | Lo reservo para simular una evaluación futura sin fuga. |

`isFraud` es mi etiqueta: es lo que quiero predecir y nunca entra como variable explicativa.

In [6]:
from modeling_utils import CATEGORICAL_FEATURES, MODEL_FEATURES, NUMERIC_FEATURES, TARGET

# Aplica transformaciones y crea variables derivadas a partir del dataset raw
# engineer_features: traduce observaciones de negocio (fraudes en TRANSFER/CASH_OUT, montos altos, vaciado de cuentas) 
# en columnas que un modelo puede interpretar (ej: log_amount, ratio saldo, error en saldo)
data = engineer_features(raw)

# Crea un dataframe con el listado de variables que usará el modelo
feature_table = pd.DataFrame({
    'variable_modelo': MODEL_FEATURES,  # Nombre de cada variable
    'tipo': ['categórica'] * len(CATEGORICAL_FEATURES) + ['numérica'] * len(NUMERIC_FEATURES),  # Clasifica si es categórica o numérica
})

# Imprime el nombre de la columna objetivo (lo que queremos predecir: isFraud)
print('Variable objetivo:', TARGET)

# Muestra la tabla de variables
feature_table

Variable objetivo: isFraud


,variable_modelo,tipo
0,type,categórica
1,amount,numérica
2,log_amount,numérica
3,oldbalanceOrg,numérica
4,newbalanceOrig,numérica
5,oldbalanceDest,numérica
6,newbalanceDest,numérica
7,origin_balance_diff,numérica
8,dest_balance_diff,numérica
9,balance_error_origin,numérica


## Separación temporal: entrenar con el pasado y evaluar el futuro

En PaySim, **`step` es la hora simulada en la que ocurre cada transacción**: `1 step` equivale a `1 hora`. Lo utilizo para ordenar los registros desde el pasado hacia el futuro.

No mezclo aleatoriamente transacciones antiguas y futuras. Si el objetivo es aproximarme a una alerta real, debo entrenar con operaciones anteriores y comprobar qué ocurre después.

Uso tres periodos:

- **Train:** `step <= 323`; aquí aprende el modelo.
- **Validation:** `324 <= step <= 378`; aquí escogeré el umbral de alerta en el notebook 05.
- **Test:** `step > 378`; lo guardaré sin tocar hasta la evaluación final del notebook 06.

Este corte deja aproximadamente 70% / 15% / 15% de filas por orden temporal y permite observar si el fraude cambia con el tiempo. `step` solo define estos periodos: **no se entrega al modelo como variable predictora**.

In [7]:
# Importa las funciones que necesitamos:
# - temporal_split(data): divide el dataset en train/validation/test según la columna 'step' (cortes definidos en el notebook).
# - split_summary(train, validation, test): construye un resumen (número de filas, fraudes, tasas, etc.) de cada partición.
from modeling_utils import split_summary, temporal_split

# Aplica la separación temporal sobre el dataframe `data`.
# Devuelve tres dataframes: train (pasado), validation (periodo intermedio) y test (futuro).
train, validation, test = temporal_split(data)

# Genera un resumen con métricas descriptivas de cada partición para inspección y registro.
summary = split_summary(train, validation, test)

# Muestra el resumen en la salida del notebook.
summary

# Por qué: realizar una separación temporal evita fugas de información (entrenar con datos futuros)
# y el resumen permite verificar tamaños y desbalance en cada partición antes de muestrear o entrenar.

,transacciones,fraudes,tasa_fraude_pct,step_min,step_max
periodo,,,,,
train,4463587,3643,0.0816,1,323
validation,980416,564,0.0575,324,378
test,918617,4006,0.4361,379,743


## Desbalance: qué haré sin falsear la prueba

El fraude es muy escaso frente a las operaciones normales. Para que entrenar sea viable en un equipo local, conservo **todos los fraudes de train** y selecciono hasta **20 operaciones normales por cada fraude** de ese mismo periodo.

La decisión importante es esta: **no reduzco validation ni test**. Así las métricas finales siguen enfrentando la distribución que realmente aparece en esos periodos.

In [9]:
from modeling_utils import training_sample

train_sample = training_sample(train)
from modeling_utils import training_sample  # Importa la función que construye una muestra balanceada para entrenamiento

train_sample = training_sample(train)  # Genera una muestra del conjunto de train conservando todos los fraudes y reduciendo las legítimas
sampling_summary = pd.Series({  # Crea un resumen con métricas básicas de la muestra resultante
    'Filas train original': len(train),  # Cuenta cuántas filas tenía el train completo
    'Fraudes train original': int(train[TARGET].sum()),  # Cuenta cuántos fraudes había en el train original
    'Filas muestra para entrenar': len(train_sample),  # Cuenta cuántas filas quedaron en la muestra usada para entrenar
    'Fraudes conservados en muestra': int(train_sample[TARGET].sum()),  # Verifica que se conservaron los fraudes en la muestra
    'Legítimas en muestra': int((train_sample[TARGET] == 0).sum()),  # Cuenta las transacciones no fraudulentas en la muestra
    'Tasa fraude muestra (%)': round(train_sample[TARGET].mean() * 100, 4),  # Calcula el porcentaje de fraude en la muestra
})

sampling_summary  # Muestra el resumen calculado
sampling_summary

Filas train original              4.463587e+06
Fraudes train original            3.643000e+03
Filas muestra para entrenar       7.650300e+04
Fraudes conservados en muestra    3.643000e+03
Legítimas en muestra              7.286000e+04
Tasa fraude muestra (%)           4.761900e+00
dtype: float64

In [10]:
summary_path = PROCESSED_DIR / 'modeling_split_summary.csv'
summary.to_csv(summary_path)
print('Resumen de particiones guardado en:', summary_path)


Resumen de particiones guardado en: /home/johanrobles/Documentos/Idea/analisis-fraude-&-modelo-predictivo-datasetPaySim/data/processed/modeling_split_summary.csv


## Conclusión del notebook

He construido variables alineadas con los hallazgos de negocio y he definido una evaluación más honesta: aprender del pasado y medir en operaciones posteriores. El hecho de que la tasa de fraude cambie entre periodos será relevante al interpretar el modelo; un buen resultado en entrenamiento no garantiza el mismo comportamiento futuro.

El siguiente paso es abrir `05_modeling.ipynb`, donde justificaré un único clasificador, lo entrenaré con la muestra de `train` y seleccionaré el umbral usando únicamente `validation`.